# AI-Driven Phishing Email Detection — End-to-End Pipeline

This notebook demonstrates the complete pipeline from raw data to explainable predictions.

**Phases covered**:
- Phase 1: Load & Clean Data
- Phase 2: Feature Engineering (TF-IDF + Metadata)
- Phase 3: Model Training (4 classifiers)
- Phase 4: Evaluation (metrics, confusion matrices, ROC curves)
- Phase 5: Model Comparison
- Phase 6: Explainability (coefficient analysis, feature importance)
- Phase 7: Single-email prediction with explanation

In [ ]:
import sys, os
sys.path.insert(0, '../src')

# Register MetadataExtractor in __main__ for pickle compatibility
from features import MetadataExtractor
import __main__
__main__.MetadataExtractor = MetadataExtractor

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print('All imports ready.')

## Phase 1: Load Cleaned Data

In [ ]:
df = pd.read_csv('../data/processed/cleaned.csv')
for c in ['sender', 'subject', 'email_text', 'cleaned_text']:
    df[c] = df[c].fillna('')

print(f'Loaded {len(df)} emails')
print(f'  Phishing: {(df.label == 1).sum()}')
print(f'  Legitimate: {(df.label == 0).sum()}')
print(f'  Columns: {list(df.columns)}')
df.head(3)

## Phase 2: Feature Engineering (Load Pre-built Pipeline)

In [ ]:
fp = joblib.load('../models/feature_pipeline.pkl')
X = fp.transform(df)
y = df['label'].values

# Feature names
tfidf = fp.named_transformers_['tfidf'].named_steps['tfidf']
tfidf_names = list(tfidf.get_feature_names_out())
meta_names = [
    'num_urls', 'has_ip_url', 'url_shortener_flag',
    'sender_domain_mismatch', 'num_exclamations', 'has_urgent_words',
    'email_length', 'num_uppercase_words', 'has_attachment_keyword',
]
all_names = tfidf_names + meta_names

print(f'Feature matrix: {X.shape[0]} emails x {X.shape[1]} features')
print(f'  5,000 TF-IDF terms + 9 metadata features')
print(f'  Sparse: {hasattr(X, "toarray")}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows')

## Phase 3: Load Trained Models (4 classifiers)

In [ ]:
model_names = ['LogisticRegression', 'RandomForestClassifier', 'MultinomialNB', 'MLPClassifier']
models = {}
for name in model_names:
    models[name] = joblib.load(f'../models/{name}.pkl')
    
for name, mdl in models.items():
    clf = mdl.named_steps.get('clf')
    if clf:
        print(f'{name}: {type(clf).__name__}')
        if hasattr(clf, 'coef_'):
            print(f'  Coefficients: {clf.coef_.shape}')
        if hasattr(clf, 'feature_importances_'):
            print(f'  Feature importances: {clf.feature_importances_.shape}')

## Phase 4: Evaluation — Metrics Table

In [ ]:
X_td = X_test.toarray() if hasattr(X_test, 'toarray') else X_test
X_trd = X_train.toarray() if hasattr(X_train, 'toarray') else X_train

results = []
for name in model_names:
    mdl = models[name]
    X_clipped_d = X_td
    if 'MultinomialNB' in name:
        X_clipped_d = np.maximum(X_td, 0)
    
    y_p = mdl.predict(X_clipped_d)
    y_pb = mdl.predict_proba(X_clipped_d)[:, 1]
    
    acc = accuracy_score(y_test, y_p)
    prec = precision_score(y_test, y_p)
    rec = recall_score(y_test, y_p)
    f1 = f1_score(y_test, y_p)
    auc = roc_auc_score(y_test, y_pb)
    
    results.append({
        'Model': name,
        'Accuracy': f'{acc:.4f}',
        'Precision': f'{prec:.4f}',
        'Recall': f'{rec:.4f}',
        'F1': f'{f1:.4f}',
        'ROC-AUC': f'{auc:.4f}',
    })
    print(f'{name:30s} F1={f1:.4f}  AUC={auc:.4f}')

pd.DataFrame(results)

## Phase 4: Confusion Matrices

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, name in zip(axes, model_names):
    mdl = models[name]
    y_p = mdl.predict(X_td)
    cm = confusion_matrix(y_test, y_p)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Phish'], yticklabels=['Legit', 'Phish'])
    ax.set_title(name)
    ax.set_ylabel('Actual' if ax == axes[0] else '')
    ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrices_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion matrices overview.')

## Phase 5: Model Comparison (from CSV)

In [ ]:
comparison = pd.read_csv('../reports/model_comparison.csv')
print('Model Comparison (from reports/model_comparison.csv):')
comparison.style.highlight_max(subset=['f1', 'roc_auc'], color='lightgreen')

## Phase 5: ROC Curves (All Models)

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 6))
for name in model_names:
    if hasattr(models[name], 'predict_proba'):
        y_pb = models[name].predict_proba(X_td)[:, 1]
        RocCurveDisplay.from_predictions(y_test, y_pb, name=name, ax=ax)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax.set_title('ROC Curves — All 4 Models')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/figures/roc_curves_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 6: Explainability — Logistic Regression Coefficients

In [ ]:
lr = models['LogisticRegression']
clf = lr.named_steps['clf']
coef = clf.coef_[0]

# Top 20 phishing-driving (positive) features
top_pos_idx = np.argsort(coef)[-20:]
top_neg_idx = np.argsort(coef)[:20]

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Phishing side
pos_names = [all_names[i] for i in top_pos_idx]
pos_vals = coef[top_pos_idx]
pos_types = ['Metadata' if i >= 5000 else 'TF-IDF' for i in top_pos_idx]
colors_pos = ['red' if t == 'Metadata' else 'orange' for t in pos_types]
axes[0].barh(range(20), pos_vals, color=colors_pos, edgecolor='black', alpha=0.8)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(pos_names, fontsize=9)
axes[0].set_title('Top 20 Phishing-Indicating Features')
axes[0].set_xlabel('Coefficient Value (positive → phishing)')
axes[0].invert_yaxis()

# Legitimate side
neg_names = [all_names[i] for i in top_neg_idx]
neg_vals = coef[top_neg_idx]
neg_types = ['TF-IDF' if i >= 5000 else 'Metadata' for i in top_neg_idx]
colors_neg = ['darkblue' if t == 'Metadata' else 'lightblue' for t in neg_types]
axes[1].barh(range(20), neg_vals, color=colors_neg, edgecolor='black', alpha=0.8)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(neg_names, fontsize=9)
axes[1].set_title('Top 20 Legitimate-Indicating Features')
axes[1].set_xlabel('Coefficient Value (negative → legitimate)')
axes[1].invert_yaxis()

plt.suptitle('Logistic Regression Coefficients — Top 20 Each Direction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/lr_split_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 5 phishing features:')
for i in range(5):
    print(f'  {pos_names[-i-1]}: {pos_vals[-i-1]:+.4f} ({pos_types[-i-1]})')
print('\nTop 5 legitimate features:')
for i in range(5):
    print(f'  {neg_names[i]}: {neg_vals[i]:+.4f} ({neg_types[i]})')

## Phase 6: Metadata Feature Separation

In [ ]:
X_trd_nb = np.maximum(X_trd, 0)  # just for access

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for j, (ax, mname) in enumerate(zip(axes.flat, meta_names)):
    phish_vals = X_trd[y_train == 1, 5000+j]
    legit_vals = X_trd[y_train == 0, 5000+j]
    ax.bar(['Legitimate', 'Phishing'],
           [np.mean(legit_vals), np.mean(phish_vals)],
           color=['#28a745', '#dc3545'], alpha=0.8, edgecolor='black')
    ax.set_title(mname, fontsize=9)
    ax.set_ylabel('Mean Value')
plt.suptitle('Metadata Feature Means by Class (Train Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/metadata_means_notebook.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 7: Single Email Prediction (Explainable)

In [ ]:
from preprocessing import clean_text, tokenize_and_lemmatize

# Example: test email
sample_email = """Subject: Urgent: Verify Your Account Now!
Dear Customer,
Your account has been compromised. Please click the link below to verify your identity:
http://192.168.1.1/verify?account=12345
Failure to do so within 24 hours will result in account suspension.
Regards,
Security Team"""

sender = "security@bank-alert.xyz"
subject = "Urgent: Verify Your Account Now!"

# Prepare
cleaned = tokenize_and_lemmatize(clean_text(sample_email))
input_df = pd.DataFrame([{
    'email_text': sample_email,
    'sender': sender,
    'subject': subject,
    'cleaned_text': cleaned,
}])

# Transform
X_in = fp.transform(input_df)
X_in_dense = X_in.toarray() if hasattr(X_in, 'toarray') else X_in

# Predict with LR
lr = models['LogisticRegression']
y_pred = lr.predict(X_in_dense)[0]
y_proba = lr.predict_proba(X_in_dense)[0]

print(f'Prediction: {"PHISHING" if y_pred == 1 else "LEGITIMATE"}')
print(f'Confidence: {y_proba[y_pred]:.2%}')
print(f'  P(phishing): {y_proba[1]:.4f}')
print(f'  P(legitimate): {y_proba[0]:.4f}')

# Top feature contributions
clf = lr.named_steps['clf']
coef = clf.coef_[0]

contributions = []
for i in range(len(all_names)):
    if X_in_dense[0, i] != 0:
        contrib = X_in_dense[0, i] * coef[i]
        if abs(contrib) > 0.001:
            contributions.append((all_names[i], contrib))

contributions.sort(key=lambda x: -abs(x[1]))

print('\nTop contributing features:')
for fname, contrib in contributions[:15]:
    direction = '→ PHISH' if contrib > 0 else '← LEGIT'
    print(f'  {fname:40s} {contrib:+.4f} {direction}')

## Phase 6: Explainability — Feature Importance (Random Forest)

In [ ]:
rf = models['RandomForestClassifier']
clf_rf = rf.named_steps['clf']
importances = clf_rf.feature_importances_

top_idx = np.argsort(importances)[-20:]
top_names = [all_names[i] for i in top_idx]
top_vals = importances[top_idx]
top_types = ['TF-IDF' if i < 5000 else 'Metadata' for i in top_idx]

colors = ['orange' if t == 'TF-IDF' else 'red' for t in top_types]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(20), top_vals, color=colors, edgecolor='black', alpha=0.85)
ax.set_yticks(range(20))
ax.set_yticklabels(top_names, fontsize=9)
ax.set_xlabel('Gini Importance')
ax.set_title('Random Forest — Top 20 Feature Importances\n(Orange=TF-IDF Term, Red=Metadata)')
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elem = [
    Patch(facecolor='orange', edgecolor='black', label='TF-IDF Term'),
    Patch(facecolor='red', edgecolor='black', label='Metadata Feature'),
]
ax.legend(handles=legend_elem, loc='lower right')

plt.tight_layout()
plt.savefig('../reports/figures/rf_importance_notebook.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary & Recommendation

| Model | F1 | AUC | Interpretability |
|-------|----|-----|------------------|
| Logistic Regression | **0.9865** | 0.9988 | **High** (linear) |
| Random Forest | 0.9837 | 0.9989 | Medium (importance) |
| MultinomialNB | 0.9427 | 0.9904 | Medium |
| MLP | 0.9866 | 0.9990 | Low (black box) |

**Recommendation**: Deploy **Logistic Regression**. It matches RF/MLP on F1 and AUC while providing full transparency through linear coefficients and LIME explanations.

**Next Steps**:
1. Run `streamlit run app/streamlit_app.py` to interact with the deployed model
2. Read `reports/explainability_notes.md` for detailed Phase 6 analysis
3. View `reports/figures/` for all generated visualizations

---
*Generated as part of: Explainable AI for Phishing Email Detection*